# ANN Index Internals — Hands-On

**LLM Engineering · Domain 1 · Roadmap Weeks 09/14**

Companion to `02 Literature Notes/LLM Engineering/ANN Index Internals` and the deck
`Lesson_06_ANN_Index_Internals.pptx`. The FAISS sections run if `faiss-cpu` is
installed; otherwise a pure-numpy fallback demonstrates the same recall/prune ideas
so the notebook always executes.

**Sources:** HNSW (arXiv:1603.09320); Product Quantization (TPAMI 2011); FAISS (arXiv:1702.08734); FAISS wiki.

## 0. Setup — a synthetic normalized corpus

In [ ]:
%pip install -q numpy
import numpy as np, time
rng = np.random.RandomState(0)
N, d = 30_000, 128
xb = rng.randn(N, d).astype("float32"); xb /= np.linalg.norm(xb, axis=1, keepdims=True)
xq = xb[:500] + 0.05*rng.randn(500, d).astype("float32"); xq /= np.linalg.norm(xq, axis=1, keepdims=True)
print("corpus:", xb.shape, " queries:", xq.shape)
try:
    import faiss; HAVE_FAISS = True
except Exception:
    HAVE_FAISS = False
print("faiss available:", HAVE_FAISS)

## 1. Exact baseline = ground truth for recall
Every ANN measurement is 'of the true top-k, how many did we return?'

In [ ]:
def flat_topk(q, mat, k=10):
    s = mat @ q
    idx = np.argpartition(-s, k)[:k]
    return idx[np.argsort(-s[idx])]

gt = np.stack([flat_topk(q, xb, 10) for q in xq])
def recall_at_10(pred):
    return float(np.mean([len(set(a.tolist()) & set(t.tolist()))/10 for a, t in zip(pred, gt)]))
print("ground-truth shape:", gt.shape)

## 2. HNSW — navigate a graph, and the efSearch dial
With faiss we build a real HNSW; without it we mimic the recall/latency tradeoff by
scanning a fraction of the corpus (a stand-in 'candidate budget').

In [ ]:
def hnsw_search(k=10, ef=64):
    if HAVE_FAISS:
        idx = faiss.IndexHNSWFlat(d, 32); idx.hnsw.efConstruction = 200
        idx.add(xb); idx.hnsw.efSearch = ef
        _, ids = idx.search(xq, k); return ids
    # fallback: candidate budget ~ ef/256 of the corpus
    frac = min(1.0, ef/256)
    sub = rng.choice(N, size=max(k, int(frac*N)), replace=False)
    return np.stack([sub[flat_topk(q, xb[sub], k)] for q in xq])

print(f"{'efSearch':>8} {'recall@10':>10}")
for ef in (8, 32, 128, 256):
    print(f"{ef:>8} {recall_at_10(hnsw_search(10, ef)):>10.3f}")

> Recall rises with `efSearch` then plateaus — pick the smallest value past the knee.

## 3. IVF — prune with clustering, and the nprobe dial
We implement a tiny IVF by hand (k-means-lite) so it runs without faiss, showing why
`nprobe=1` misses neighbors near cell boundaries.

In [ ]:
# learn nlist centroids by sampling, assign each vector to nearest
nlist = 64
cent = xb[rng.choice(N, nlist, replace=False)].copy()
assign = np.argmax(xb @ cent.T, axis=1)          # nearest centroid per vector
lists = {c: np.where(assign == c)[0] for c in range(nlist)}

def ivf_search(q, k=10, nprobe=1):
    probe = np.argsort(-(cent @ q))[:nprobe]       # nearest nprobe centroids
    cand = np.concatenate([lists[c] for c in probe]) if nprobe else np.array([],int)
    if len(cand) < k: return cand
    local = flat_topk(q, xb[cand], k)
    return cand[local]

print(f"{'nprobe':>6} {'recall@10':>10}")
for npb in (1, 4, 16, 64):
    pred = np.stack([np.pad(ivf_search(q,10,npb), (0,10))[:10] for q in xq])
    print(f"{npb:>6} {recall_at_10(pred):>10.3f}")

> `nprobe=1` is fast but low recall (boundary misses); `nprobe=nlist` = exact.

## 4. PQ — compress each vector, then re-rank
Product quantization replaces each vector with m sub-space codes. Here m sub-quantizers
of 256 centroids each. Note how raw PQ ranking is lossy and a re-rank fixes it.

In [ ]:
m, ksub = 8, 256
dsub = d // m
subq = []                      # per-subspace codebooks
codes = np.zeros((N, m), dtype=np.int32)
for s in range(m):
    sl = slice(s*dsub, (s+1)*dsub)
    cb = xb[rng.choice(N, ksub, replace=False), sl].copy()   # codebook
    codes[:, s] = np.argmax(xb[:, sl] @ cb.T, axis=1)
    subq.append(cb)

def pq_search(q, k=10, rerank=0):
    # asymmetric distance via lookup tables, summed over subspaces
    score = np.zeros(N)
    for s in range(m):
        sl = slice(s*dsub, (s+1)*dsub)
        lut = subq[s] @ q[sl]              # query vs each centroid
        score += lut[codes[:, s]]
    cand = np.argpartition(-score, max(k, rerank or k))[:max(k, rerank or k)]
    if rerank:                              # exact re-rank of top candidates
        cand = cand[np.argsort(-(xb[cand] @ q))]
    else:
        cand = cand[np.argsort(-score[cand])]
    return cand[:k]

raw   = np.stack([pq_search(q, 10, rerank=0)   for q in xq])
rrank = np.stack([pq_search(q, 10, rerank=100) for q in xq])
print(f"PQ raw        recall@10 = {recall_at_10(raw):.3f}")
print(f"PQ + re-rank  recall@10 = {recall_at_10(rrank):.3f}")
print(f"bytes/vector: PQ={m} vs float32={d*4}  -> {d*4//m}x smaller")

## 5. Exercises
1. In §2 push `efSearch` higher — where does recall stop improving?
2. In §3 raise `nlist` to 256 and re-check recall at `nprobe=4`. Why does it change?
3. In §4 try `m=16` and `m=4`. Trade-off between compression and recall?
4. Add timing to each section and plot recall vs ms/query for HNSW.
5. Estimate the RAM for 10M x 768-d under HNSW (M=32) vs IVFPQ (m=64). Which fits 16 GB?

## Links
- Literature note: `02 Literature Notes/LLM Engineering/ANN Index Internals`
- Snippets: `04 Code Snippets/LLM/HNSW and IVF with FAISS`, `.../Tuning ANN Recall vs Latency`
- MOC: `06 Maps of Content/LLM Engineering Concepts`